## Session 25 - Iris Dataset

#### Q1. Manual Hyperparameter Tuning – KNN

In [109]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [110]:
iris = load_iris()

X = iris.data
y = iris.target

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [111]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 100
Testing Samples: 50


In [112]:
k_values = [3, 5, 7, 11, 13, 15]

scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)
    scores.append(score)
    print(f"K = {k} --> Accuracy = {score:.4f}")

K = 3 --> Accuracy = 0.9800
K = 5 --> Accuracy = 0.9800
K = 7 --> Accuracy = 0.9800
K = 11 --> Accuracy = 1.0000
K = 13 --> Accuracy = 1.0000
K = 15 --> Accuracy = 1.0000


In [113]:
best_score = max(scores)

best_k = k_values[scores.index(best_score)]

print("Best K Value :", best_k)
print("Highest Accuracy :", round(best_score, 4))

Best K Value : 11
Highest Accuracy : 1.0


In [114]:
print("Summary:")
print(f"The highest accuracy was obtained when n_neighbors = {best_k}.")
print("This K value is selected as the optimal hyperparameter for the KNN model.")

Summary:
The highest accuracy was obtained when n_neighbors = 11.
This K value is selected as the optimal hyperparameter for the KNN model.


#### Manual Hyperparameter Tuning – SVM

In [115]:
from sklearn.svm import SVC

In [116]:
c_values = [1, 10, 20]
kernels = ["linear", "rbf"]

results = []

for c in c_values:
    for kernel in kernels:
        model = SVC(C=c, kernel=kernel)
        model.fit(X_train, y_train)

        accuracy = model.score(X_test, y_test)
        results.append([c, kernel, accuracy])
        print(f"C = {c}, Kernel = {kernel} --> Accuracy = {accuracy:.4f}")

C = 1, Kernel = linear --> Accuracy = 1.0000
C = 1, Kernel = rbf --> Accuracy = 1.0000
C = 10, Kernel = linear --> Accuracy = 1.0000
C = 10, Kernel = rbf --> Accuracy = 0.9800
C = 20, Kernel = linear --> Accuracy = 0.9800
C = 20, Kernel = rbf --> Accuracy = 1.0000


In [117]:
comparison = pd.DataFrame(
    results,
    columns=["C", "Kernel", "Accuracy"]
)

print(comparison)

    C  Kernel  Accuracy
0   1  linear      1.00
1   1     rbf      1.00
2  10  linear      1.00
3  10     rbf      0.98
4  20  linear      0.98
5  20     rbf      1.00


In [118]:
best_result = comparison.loc[
    comparison["Accuracy"].idxmax()
]

print("\nBest Hyperparameters")
print("C :", best_result["C"])
print("Kernel :", best_result["Kernel"])
print("Accuracy :", round(best_result["Accuracy"], 4))


Best Hyperparameters
C : 1
Kernel : linear
Accuracy : 1.0


In [119]:
print("\nSummary")
print(f"The highest accuracy was achieved using C = {best_result['C']} and Kernel = {best_result['Kernel']}.")
print("This combination is selected as the best hyperparameter setting for the SVM model.")


Summary
The highest accuracy was achieved using C = 1 and Kernel = linear.
This combination is selected as the best hyperparameter setting for the SVM model.


#### Q3. Grid Search CV

In [120]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

In [121]:
param_grid = {
    "C": [1, 10, 20],
    "kernel": ["linear", "rbf"]
}

In [122]:
svm_grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

svm_grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [1, 10, 20], 'kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [123]:
results = pd.DataFrame(svm_grid.cv_results_)

comparison = results[
    ["param_C", "param_kernel", "mean_test_score"]
]

print(comparison)

   param_C param_kernel  mean_test_score
0        1       linear             0.95
1        1          rbf             0.93
2       10       linear             0.93
3       10          rbf             0.94
4       20       linear             0.93
5       20          rbf             0.95


In [124]:
print("Best Parameters:", svm_grid.best_params_)
print("Best Score:", round(svm_grid.best_score_, 4))

Best Parameters: {'C': 1, 'kernel': 'linear'}
Best Score: 0.95


In [125]:
print("\nSummary")
print("GridSearchCV tested all combinations of C and kernel using 5-fold cross-validation.")
print("The combination with the highest mean test score was selected as the best model.")


Summary
GridSearchCV tested all combinations of C and kernel using 5-fold cross-validation.
The combination with the highest mean test score was selected as the best model.


#### Q4. Randomized Search CV

In [126]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC

In [127]:
param_grid = {
    "C": [1, 10, 20],
    "kernel": ["linear", "rbf"]
}

In [128]:
random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=param_grid,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    random_state=42
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=SVC(), n_iter=5,
                   param_distributions={'C': [1, 10, 20],
                                        'kernel': ['linear', 'rbf']},
                   random_state=42, scoring='accuracy')

In [129]:
results = pd.DataFrame(random_search.cv_results_)

comparison = results[
    ["param_C", "param_kernel", "mean_test_score"]
]

print(comparison)

   param_C param_kernel  mean_test_score
0        1       linear             0.95
1        1          rbf             0.93
2       20          rbf             0.95
3       10       linear             0.93
4       20       linear             0.93


In [130]:
print("Best Parameters:", random_search.best_params_)
print("Best Score:", round(random_search.best_score_, 4))

Best Parameters: {'kernel': 'linear', 'C': 1}
Best Score: 0.95


In [131]:
print("Grid Search Best Parameters :", grid_search.best_params_)
print("Grid Search Best Score :", round(grid_search.best_score_, 4))
print()
print("Random Search Best Parameters :", random_search.best_params_)
print("Random Search Best Score :", round(random_search.best_score_, 4))

Grid Search Best Parameters : {'max_depth': 3, 'n_estimators': 50}
Grid Search Best Score : 0.9583

Random Search Best Parameters : {'kernel': 'linear', 'C': 1}
Random Search Best Score : 0.95


In [132]:
print("\nSummary")
print("RandomizedSearchCV evaluated 5 randomly selected parameter combinations using 5-fold cross-validation.")
print("GridSearchCV evaluates all possible parameter combinations, whereas RandomizedSearchCV evaluates only a subset, making it faster for larger search spaces.")
print("The best parameters and scores from both methods are compared above.")


Summary
RandomizedSearchCV evaluated 5 randomly selected parameter combinations using 5-fold cross-validation.
GridSearchCV evaluates all possible parameter combinations, whereas RandomizedSearchCV evaluates only a subset, making it faster for larger search spaces.
The best parameters and scores from both methods are compared above.


#### Q5. Bagging – Random Forest

In [133]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [134]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [135]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [136]:
y_pred = rf_model.predict(X_test)

In [137]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", round(accuracy, 4))

Accuracy Score: 0.9


In [138]:
print("\nSummary")
print(f"The Random Forest Classifier achieved an accuracy of {accuracy:.4f} on the test dataset.")


Summary
The Random Forest Classifier achieved an accuracy of 0.9000 on the test dataset.


#### Q6. Boosting – AdaBoost & Gradient Boosting

In [139]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

In [140]:
ada_model = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

ada_model.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=100, random_state=42)

In [141]:
y_pred_ada = ada_model.predict(X_test)

In [142]:
ada_accuracy = accuracy_score(y_test, y_pred_ada)
print("AdaBoost Accuracy:", round(ada_accuracy, 4))

AdaBoost Accuracy: 0.9333


In [143]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb_model.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [144]:
y_pred_gb = gb_model.predict(X_test)

In [145]:
gb_accuracy = accuracy_score(y_test, y_pred_gb)
print("Gradient Boosting Accuracy:", round(gb_accuracy, 4))

Gradient Boosting Accuracy: 0.9667


In [146]:
print("\nSummary")
print(f"AdaBoost Accuracy: {ada_accuracy:.4f}")
print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")


Summary
AdaBoost Accuracy: 0.9333
Gradient Boosting Accuracy: 0.9667


#### Q7. Boosting – XGBoost

In [147]:
from xgboost import XGBClassifier

In [148]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [149]:
y_pred_xgb = xgb_model.predict(X_test)

In [150]:
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
print("XGBoost Accuracy:", round(xgb_accuracy, 4))

XGBoost Accuracy: 0.9333


In [151]:
print("\nSummary")
print(f"The XGBoost Classifier achieved an accuracy of {xgb_accuracy:.4f} on the test dataset.")


Summary
The XGBoost Classifier achieved an accuracy of 0.9333 on the test dataset.


#### Q8. Hyperparameter Tuning on Random Forest

In [152]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [3, 5, 7]
}

In [153]:
rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

rf_grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 7],
                         'n_estimators': [50, 100, 150]},
             scoring='accuracy')

In [154]:
results = pd.DataFrame(rf_grid.cv_results_)

comparison = results[
    ["param_n_estimators", "param_max_depth", "mean_test_score"]
]

print(comparison)

   param_n_estimators  param_max_depth  mean_test_score
0                  50                3         0.958333
1                 100                3         0.958333
2                 150                3         0.958333
3                  50                5         0.950000
4                 100                5         0.950000
5                 150                5         0.950000
6                  50                7         0.950000
7                 100                7         0.950000
8                 150                7         0.950000


In [155]:
print("Best Parameters:", rf_grid.best_params_)
print("Best Score:", round(rf_grid.best_score_, 4))

Best Parameters: {'max_depth': 3, 'n_estimators': 50}
Best Score: 0.9583


In [156]:
print("\nSummary")
print("GridSearchCV evaluated all combinations of n_estimators and max_depth using 5-fold cross-validation.")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Score: {grid_search.best_score_:.4f}")


Summary
GridSearchCV evaluated all combinations of n_estimators and max_depth using 5-fold cross-validation.
Best Parameters: {'max_depth': 3, 'n_estimators': 50}
Best Score: 0.9583


#### Q9. Complete Model Comparison

In [158]:
svm_best = SVC(
    C=1,
    kernel="linear"
)

svm_best.fit(X_train, y_train)
y_pred_svm = svm_best.predict(X_test)
svm_accuracy = accuracy_score(y_test, y_pred_svm)

In [159]:
y_pred_rf = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

In [160]:
y_pred_ada = ada_model.predict(X_test)
ada_accuracy = accuracy_score(y_test, y_pred_ada)

In [161]:
y_pred_gb = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred_gb)

In [162]:
y_pred_xgb = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

In [163]:
comparison = pd.DataFrame({
    "Model": [
        "SVM",
        "Random Forest",
        "AdaBoost",
        "Gradient Boosting",
        "XGBoost"
    ],
    "Accuracy Score": [
        svm_accuracy,
        rf_accuracy,
        ada_accuracy,
        gb_accuracy,
        xgb_accuracy
    ]
})

comparison = comparison.sort_values(
    by="Accuracy Score",
    ascending=False
)

comparison.reset_index(drop=True, inplace=True)
print(comparison)

               Model  Accuracy Score
0                SVM        1.000000
1  Gradient Boosting        0.966667
2           AdaBoost        0.933333
3            XGBoost        0.933333
4      Random Forest        0.900000


In [164]:
best_model = comparison.iloc[0]

print("\nBest Model :", best_model["Model"])
print("Accuracy Score :", round(best_model["Accuracy Score"], 4))


Best Model : SVM
Accuracy Score : 1.0


In [165]:
print("\nSummary")
print(f"The best performing model is {best_model['Model']} with an accuracy score of {best_model['Accuracy Score']:.4f}.")


Summary
The best performing model is SVM with an accuracy score of 1.0000.
